# 🤘 Cache Me If You Can — Act III: the levers

It is easy to picture a language model as one conversation between a person and the model: a single **user**. In reality, many conversations run at the same time, for example:

- two people chatting with the same model at once,
- a coding **agent** (a program that calls the model in a loop to get a task done) that spawns several sub-agents, each with its own conversation,
- a script summarising a thousand documents.

Each user has its own data on the card: its own activations and its own KV cache. Serving them means answering two questions: **how many users can one card serve at the same time**, and **how many tokens per second does each of them get**, and all of them together? Two settings decide it: the **batch size** and the **number format** of the weights. You will change each one and measure what happens.

This act goes back to the model of Act I, run with **tt_transformers**: Tenstorrent's open-source TT-NN implementations of popular language models (Llama, Qwen, DeepSeek and others), which ship with tt-metal. It takes care of serving many users at once, their KV caches, trace and number formats, so each of those is a single setting here.

**Why there is a budget.** A fast reader reads about 5 words per second, roughly 6 to 7 tokens per second, or one token every 150 ms. If tokens arrive slower than that, people sit watching the cursor. Products aim well above reading speed, for two reasons: reasoning models like ours write hundreds of "thinking" tokens before the visible answer, and an agent chains many calls one after another, so every millisecond per token is paid many times over. A common target is 20 tokens per second per user, one token every **50 ms**. For the first token, a chat window feels stuck after about **2 seconds**. Both numbers live in `common/config.py`.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "2026" / "common").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "2026"))
from common import config, wait_for_warmup
wait_for_warmup()
import json, os, random
import matplotlib.pyplot as plt
from common.llm import LLM

PROMPTS = ["Explain what a KV cache is to a first-year student.",
           "Why is the first token of a chatbot's answer slower than the rest?",
           "Give three reasons memory bandwidth matters for generating text.",
           "What does batching buy an inference server, and what does it cost?",
           "Describe grouped-query attention in two sentences.",
           "Why do serving systems store the KV cache in pages?",
           "What is speculative decoding, in one paragraph?",
           "What changes when a model no longer fits on one chip?"]
print(f"budget: at most {config.TOKEN_BUDGET_MS:.0f} ms per token per user, "
      f"first token within {config.TTFT_BUDGET_MS / 1000:.0f} s")

## 1. Batch size

A **batch** is a group of users processed together. One decode step produces one new token for **every** user in the batch.

Why it matters: without batching, each decode step would serve a single user. Every step reads all the weights from memory (Act I), so 8 users would mean 8 full reads of the weights for 8 tokens, while the others wait their turn. With a batch of 8, one read of the weights produces a token for all 8.

**Fill in the loop.** For each batch size: create the model for that many users (the warm-up already prepared these sizes, so loading takes seconds), generate `config.ACT3_NEW_TOKENS` tokens for all of them with `stop_at_eos=False`, record the milliseconds per token and the total tokens per second, and close the model before the next size.

In [ ]:
results = {}
for batch_size in config.ACT3_BATCH_SIZES:
    # >>> YOUR CODE HERE: the same pattern as measure() in Act I; only one model fits on the card at a time, so close each one before loading the next
    raise NotImplementedError("the same pattern as measure() in Act I; only one model fits on the card at a time, so close each one before loading the next")
    # <<<
    print(f"{batch_size:>3} users: {results[batch_size]['ms_per_token']:6.1f} ms per token per user, "
          f"{results[batch_size]['tokens_per_s_total']:7.0f} tokens per second in total")

In [ ]:
sizes = sorted(results)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(sizes, [results[b]["ms_per_token"] for b in sizes], "o-")
ax[0].axhline(config.TOKEN_BUDGET_MS, ls="--", c="gray", label=f"budget: {config.TOKEN_BUDGET_MS:.0f} ms")
ax[0].set(xlabel="users in the batch", ylabel="ms per token, per user", title="What each user sees"); ax[0].legend()
ax[1].plot(sizes, [results[b]["tokens_per_s_total"] for b in sizes], "s-")
ax[1].set(xlabel="users in the batch", ylabel="tokens per second, all users", title="What the card produces")
for a in ax: a.grid(alpha=0.3); a.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

Each user's speed changed far less than the number of users did, so the total grew almost in proportion.

Why: in Act I you saw that a decode step spends its time reading the weights. With 8 users in the batch, that one read of the weights serves 8 tokens instead of 1, so the step costs only a little more while producing 8 times as many tokens.

This stops at some batch size. Every user adds arithmetic (and every user's KV cache has to be read too). Once that work takes longer than reading the weights, each extra user adds its own time, and the per-user line starts to climb.

## 2. Number format

The weights could be stored as classic 32-bit floating-point numbers, but they would take a lot of space, and every decode step would move 4 bytes per weight. Machine learning tolerates less precision, so smaller formats are used. Three appear in this model:

- **`bfloat16`**: 16 bits per number: 1 sign bit, 8 exponent bits, 7 fraction (mantissa) bits. The same range as a 32-bit float, with less precision.
- **`bfloat8_b`**, a *block floating-point* format: numbers are stored in blocks of 16 that share one 8-bit exponent; each number keeps its own sign and 7 mantissa bits. About 8.5 bits per number, half of `bfloat16`.
- **`bfloat4_b`**: the same idea with 3 mantissa bits, about 4.5 bits per number.

Fewer bits per weight means fewer bytes to read per token, and decode is limited by reading. The price is precision: inside a block, numbers much smaller than the largest one lose detail.

tt_transformers has two presets for this model:

- **accuracy**: attention weights and KV cache in `bfloat16`, feed-forward weights in `bfloat8_b`.
- **performance**: attention weights and KV cache in `bfloat8_b`, the two largest feed-forward matrices in `bfloat4_b`.

The cell below runs each preset once without timing (to finish any setup), then once timed, and prints how many bytes each decode step reads.

In [ ]:
prompt = PROMPTS[0]
answers = {}
for preset in ("accuracy", "performance"):
    llm = LLM(max_batch_size=1, preset=preset)
    llm.generate(prompt, max_new_tokens=8)                                     # untimed: finish setup
    r = llm.generate(prompt, max_new_tokens=config.ACT3_NEW_TOKENS, stop_at_eos=False)
    gb = llm.facts["weight_bytes_per_token"] / 1e9
    answers[preset] = r.texts[0]
    print(f"{preset:>11}: reads {gb:.2f} GB per token, {r.decode_ms_per_token:5.1f} ms per token "
          f"({r.tokens_per_s_per_user:5.1f} tokens/s)")
    llm.close()

order = list(answers); random.shuffle(order)
for label, preset in zip("AB", order):
    print(f"\n--- answer {label} ---\n{answers[preset][:600]}")
reveal = dict(zip("AB", order))

Compare the two changes: how much did the bytes per token drop, and how much did the time per token drop? On this small model the time drops by less than the bytes, because reading weights is only part of each step (Act I showed the rest). On larger models the two move closer together.

**Vote before you look:** which answer came from the smaller numbers? Then run the next cell.

In [ ]:
reveal

## 3. How many users fit inside the budget?

**Fill in `users_within_budget`.** Among the batch sizes you measured in section 1, find the largest one whose milliseconds per token stay within the budget, and return it with its total tokens per second.

In [ ]:
def users_within_budget(results, budget_ms):
    """Returns (largest batch size within budget, its total tokens/s), or (0, 0.0) if none fits."""
    # >>> YOUR CODE HERE: first find the batch sizes that stay within the budget, then pick the best of those
    raise NotImplementedError("first find the batch sizes that stay within the budget, then pick the best of those")
    # <<<

users, total = users_within_budget(results, config.TOKEN_BUDGET_MS)
print(f"{users} users within {config.TOKEN_BUDGET_MS:.0f} ms per token, {total:.0f} tokens per second in total")

**Free the card for the next act.** A program that has used the card keeps it until it stops, so the last cell stops this notebook's kernel. Your outputs stay on the page; VS Code will show the kernel as stopped, which is expected.

In [ ]:
print("Act III done.")
from common import release_card
release_card()